# Aşama 2 baseline: TR-Omni-E4B ve emotion2vec+ (eğitimsiz)

İnce ayardan önce çıtayı ölçer. Eğitim yok; modeller hazır hâliyle çalışır.

| Bölüm | Veri | Ne ölçülür |
|---|---|---|
| `e2v_algi` | TurEV-DB, sınıf başına 40 kayıt (160) | emotion2vec+ base hazır etiketi: 4 duygu UAR, "kızgın mı?" UAR |
| `omni_algi` | Aynı 160 kayıt | TR-Omni'ye sesle birlikte "duygu hangisi?" sorulur: aynı ölçütler + tanınamayan cevap |
| `omni_ofke` | 20 ElevenLabs müşteri tepkisi | TR-Omni'nin 0-4 öfke puanı ile mevcut hattın ölçtüğü öfkenin sıra korelasyonu |
| `omni_sohbet` | Aynı 20 tepki | TR-Omni sesli cevap: süre, Türkçe yazım denetimi, cevap sesinin duygusu; sesler dinlenebilir |

**Çalışma zamanı:** Çalışma zamanı → Çalışma zamanı türünü değiştir → **L4** veya **A100**. TR-Omni ~17 GB GPU belleği
ister; T4 (16 GB) yetmez. Model indirmesi ~21 GB, ilk çağrıda ~1 dk derleme.

**Drive hazırlığı (bir kez):** `MyDrive/DuyguAvcisi/enjeksiyon_ses/` klasörüne yerel bilgisayardaki
`data/enjeksiyon_ses/` içeriğini (20 `.wav` + `sonuc.jsonl`) yükleyin. Bu sesler git'e girmiyor; ElevenLabs ile üretilmiş
sentetik seslerdir, kişisel veri içermez. Sonuçlar `MyDrive/DuyguAvcisi/s2s_baseline/` altına yazılır; Colab koparsa
hücreyi yeniden çalıştırın, kaldığı yerden devam eder.

In [ ]:
# 1) Drive, kod ve veri
import os
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/DuyguAvcisi"
REPO = "https://github.com/Pappates/Emotional_Feedback_Costumer_Service"
DAL = "asama2/s2s-baseline"
# Depo özelse: from getpass import getpass; REPO = REPO.replace("https://", f"https://{getpass('GitHub token: ')}@")
if not os.path.exists("proje"):
    !git clone -q -b {DAL} {REPO} proje
else:
    !cd proje && git pull -q
if not os.path.exists("turev"):
    !git clone -q --depth 1 https://github.com/Xeonen/TurEV-DB turev
MUSTERI = f"{DRIVE}/enjeksiyon_ses"
print("Müşteri sesleri:", len([f for f in os.listdir(MUSTERI) if f.endswith(".wav")]) if os.path.isdir(MUSTERI) else "YOK, Drive'a yükleyin")

In [ ]:
# 2) Paketler (TR-Omni: transformers>=5.17, voxcpm==2.0.3, PyTorch 2.8+)
!pip -q install "transformers>=5.17" voxcpm==2.0.3 funasr librosa soundfile zeyrek wordfreq
import torch
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "YOK")
if torch.cuda.is_available():
    print(f"GPU belleği: {torch.cuda.get_device_properties(0).total_memory / 2**30:.0f} GB (en az ~17 GB gerekli)")

In [ ]:
# 3) TR-Omni-E4B indir (~21 GB, Apache-2.0)
from huggingface_hub import snapshot_download
MODEL = snapshot_download("furkanbekmezci/TR-Omni-E4B-Turkish-Native-Speech-to-Speech-Model", local_dir="/content/tr_omni")
print(MODEL)

In [ ]:
# 4) Ölçüm (tümü ~30-60 dk). Yalnızca bir bölüm için: --bolum omni_sohbet
CIKTI = f"{DRIVE}/s2s_baseline"
!cd proje && python egitim/s2s_baseline.py --turev "/content/turev/Sound Source" --model {MODEL} --musteri {MUSTERI} --cikti {CIKTI} --adet 40 2>&1 | grep -v "Warning"

## Nasıl okunur?

* **uar4**: 4 duyguda dengeli doğruluk, şans 0,25. Karşılaştırma: Aşama 1'de emotion2vec+ base hazır tahmini tüm TurEV'de
  0,79 (aynı veri, aynı etiket eşlemesi).
* **kizgin_mi_uar**: "kızgın mı, değil mi" dengeli doğruluğu, şans 0,50. Projenin asıl sorusu bu.
* **taninamayan_cevap**: TR-Omni dört kelimeden birini söylemediyse yanlış sayılır; yüksekse sorun algı değil yönerge
  izleme olabilir (`omni_algi.jsonl` içindeki `cevap` alanına bakın).
* **spearman_olculen_ofke**: TR-Omni'nin öfke sıralaması mevcut hattın ölçümüyle ne kadar örtüşüyor (n=20, kaba gösterge;
  mevcut ölçüm de hatalı olabilir, gerçek etiket değildir).
* **omni_sohbet**: `ort_sure` cevabın tamamının üretilme süresi (akış yok; canlı uygulamada ilk ses ~0,7 sn bildiriliyor).
  `cevap_sesi_duygu_dagilimi` kızgın müşteriye asistanın hangi tonda duyulduğunu gösterir. Sesleri aşağıda dinleyin.

Bu sayılar ince ayarın geçmesi gereken çıtadır: `ozet.json` dosyasını saklayın.

In [ ]:
# 5) Özet ve örnek cevaplar
import json
from IPython.display import Audio, display
print(json.dumps(json.load(open(f"{CIKTI}/ozet.json")), ensure_ascii=False, indent=1))
for l in list(open(f"{CIKTI}/omni_sohbet.jsonl", encoding="utf-8"))[:4]:
    r = json.loads(l)
    print()
    print(f"Müşteri (öfke {r['musteri_ofke']}): {r['musteri']}")
    print(f"TR-Omni ({r['cevap_sesi_duygu']}): {r['cevap']}")
    display(Audio(f"{CIKTI}/sohbet_ses/{r['anahtar']}.wav"))